In [5]:
# BINARY CLASSIFICATION 

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import wandb
import numpy as np
from sklearn.metrics import confusion_matrix

# Initialize wandb
wandb.init(
    project="resnet-DFD-vs-real",
    name="ResNet50, Epochs=100, BatchSize=32, LR=1e-3, WeightDecay=1e-4",
    config={
        "epochs": 100,
        "batch_size": 32,
        "lr": 1e-3,  
        "optimizer": "Adam",
        "architecture": "resnet50",  
        "img_size": 224,
        "split": [0.7, 0.15, 0.15],
        "weight_decay": 1e-4  
    }
)
config = wandb.config

# Device
device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')

# Transforms
transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset
dataset = datasets.ImageFolder(root='fingerprints/FaceForensics++/DFD-real', transform=transform)
print("Class mapping:", dataset.class_to_idx)

# Train/Val/Test split
total_size = len(dataset)
train_size = int(config.split[0] * total_size)
val_size = int(config.split[1] * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])
val_indices = val_ds.indices
test_indices = test_ds.indices

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# Model: Changed to resnet50 for higher capacity
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# Loss and optimizer (Added L2 regularization via weight_decay)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

# Store validation misclassified info
val_misclassified_indices = []
val_misclassified_paths = []

# Training loop
for epoch in range(config.epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * train_correct / total
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(val_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            # Track misclassifications in validation
            for i in range(len(labels)):
                global_index = val_indices[batch_idx * config.batch_size + i]
                if preds[i] != labels[i]:
                    val_misclassified_indices.append(global_index)
                    img_path, _ = dataset.samples[global_index]
                    val_misclassified_paths.append(img_path)

    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)

    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), "resnet_model.pth")
wandb.save("resnet_model.pth")

# Save validation misclassifications
with open("val_misclassified_indices.txt", "w") as f:
    for idx in val_misclassified_indices:
        f.write(f"{idx}\n")

with open("val_misclassified_paths.txt", "w") as f:
    for path in val_misclassified_paths:
        f.write(f"{path}\n")

print(f"Misclassified validation indices saved ")
print(f"Misclassified validation paths saved ")

# Final test evaluation
def evaluate_on_test(model, test_loader):
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss = 0.0

    all_preds = []
    all_labels = []
    misclassified_indices = []
    misclassified_paths = []

    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(test_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(labels)):
                global_index = batch_idx * test_loader.batch_size + i
                if preds[i] != labels[i]:
                    misclassified_indices.append(global_index)
                    dataset_index = test_indices[global_index]
                    img_path, _ = dataset.samples[dataset_index]
                    misclassified_paths.append(img_path)

            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = 100.0 * test_correct / test_total
    avg_test_loss = test_loss / len(test_loader)

    print(f"\n Final Test Accuracy: {test_acc:.2f}%, Test Loss: {avg_test_loss:.4f}")
    wandb.log({
        "test_accuracy": test_acc,
        "test_loss": avg_test_loss
    })

    # Confusion Matrix in WandB
    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_labels,
            preds=all_preds,
            class_names=list(dataset.class_to_idx.keys())
        )
    })

    # Save misclassified indices and paths
    with open("test_misclassified_indices.txt", "w") as f:
        for idx in misclassified_indices:
            f.write(f"{idx}\n")

    with open("test_misclassified_paths.txt", "w") as f:
        for path in misclassified_paths:
            f.write(f"{path}\n")

    print(f" Misclassified test indices saved")
    print(f" Misclassified test paths saved")

# Run test evaluation
evaluate_on_test(model, test_loader)

Class mapping: {'DeepFakeDetection': 0, 'Real': 1}


/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


RuntimeError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# MULTI-CLASS CLASSIFICATION

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import wandb
import numpy as np
from sklearn.metrics import confusion_matrix

# Initialize wandb
wandb.init(
    project="resent-real-veo3-sora",
    name=" Epochs: 20, Batch Size: 32, LR: 1e-4",
    config={
        "epochs": 20,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "Adam",
        "architecture": "resnet18",
        "img_size": 224,
        "split": [0.7, 0.15, 0.15]
    }
)
config = wandb.config

# Device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Transforms
transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset
dataset = datasets.ImageFolder(root='fingerprints/real-sora-veo3', transform=transform)
print("Class mapping:", dataset.class_to_idx)

# Train/Val/Test split
total_size = len(dataset)
train_size = int(config.split[0] * total_size)
val_size = int(config.split[1] * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])
val_indices = val_ds.indices
test_indices = test_ds.indices

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# Model
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 3)  # 🔁 Change to 3-class output
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr)

# Track validation misclassifications
val_misclassified_indices = []
val_misclassified_paths = []

# Training loop
for epoch in range(config.epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * train_correct / total
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(val_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            # Track misclassifications in validation
            for i in range(len(labels)):
                global_index = val_indices[batch_idx * config.batch_size + i]
                if preds[i] != labels[i]:
                    val_misclassified_indices.append(global_index)
                    img_path, _ = dataset.samples[global_index]
                    val_misclassified_paths.append(img_path)

    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)

    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), "resnet_model.pth")
wandb.save("resnet_model.pth")

# Save validation misclassifications
with open("val_misclassified_indices.txt", "w") as f:
    for idx in val_misclassified_indices:
        f.write(f"{idx}\n")

with open("val_misclassified_paths.txt", "w") as f:
    for path in val_misclassified_paths:
        f.write(f"{path}\n")

print(f" Misclassified validation indices saved to 'val_misclassified_indices.txt'")
print(f" Misclassified validation paths saved to 'val_misclassified_paths.txt'")


# Final test evaluation
def evaluate_on_test(model, test_loader):
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss = 0.0

    all_preds = []
    all_labels = []
    misclassified_indices = []
    misclassified_paths = []

    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(test_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(labels)):
                global_index = batch_idx * test_loader.batch_size + i
                if preds[i] != labels[i]:
                    misclassified_indices.append(global_index)
                    dataset_index = test_indices[global_index]
                    img_path, _ = dataset.samples[dataset_index]
                    misclassified_paths.append(img_path)

            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = 100.0 * test_correct / test_total
    avg_test_loss = test_loss / len(test_loader)

    print(f"\n Final Test Accuracy: {test_acc:.2f}%, Test Loss: {avg_test_loss:.4f}")
    wandb.log({
        "test_accuracy": test_acc,
        "test_loss": avg_test_loss
    })

    # Confusion Matrix in WandB
    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_labels,
            preds=all_preds,
            class_names=list(dataset.class_to_idx.keys())
        )
    })

    # Save misclassified indices and paths
    with open("test_misclassified_indices.txt", "w") as f:
        for idx in misclassified_indices:
            f.write(f"{idx}\n")

    with open("test_misclassified_paths.txt", "w") as f:
        for path in misclassified_paths:
            f.write(f"{path}\n")

    print(f" Misclassified test indices saved to 'test_misclassified_indices.txt'")
    print(f" Misclassified test paths saved to 'test_misclassified_paths.txt'")


# Run test evaluation
evaluate_on_test(model, test_loader)


wandb: Currently logged in as: aarushishukla (aarushishukla-uc-santa-barbara) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Class mapping: {'real': 0, 'sora': 1, 'veo3': 2}


/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1: Train Loss: 1.0470, Train Acc: 52.24%, Val Loss: 1.0482, Val Acc: 28.57%
Epoch 2: Train Loss: 0.5557, Train Acc: 77.61%, Val Loss: 1.0541, Val Acc: 50.00%
Epoch 3: Train Loss: 0.3595, Train Acc: 88.06%, Val Loss: 1.1006, Val Acc: 42.86%
Epoch 4: Train Loss: 0.3705, Train Acc: 88.06%, Val Loss: 1.1269, Val Acc: 42.86%
Epoch 5: Train Loss: 0.3265, Train Acc: 88.06%, Val Loss: 1.1644, Val Acc: 42.86%
Epoch 6: Train Loss: 0.6003, Train Acc: 86.57%, Val Loss: 1.2696, Val Acc: 50.00%
Epoch 7: Train Loss: 0.4248, Train Acc: 86.57%, Val Loss: 1.2504, Val Acc: 50.00%
Epoch 8: Train Loss: 0.3986, Train Acc: 83.58%, Val Loss: 1.2211, Val Acc: 50.00%
Epoch 9: Train Loss: 0.2755, Train Acc: 88.06%, Val Loss: 1.2252, Val Acc: 57.14%
Epoch 10: Train Loss: 0.3582, Train Acc: 85.07%, Val Loss: 1.3285, Val Acc: 50.00%
Epoch 11: Train Loss: 0.3727, Train Acc: 86.57%, Val Loss: 1.3922, Val Acc: 50.00%
Epoch 12: Train Loss: 0.3983, Train Acc: 86.57%, Val Loss: 1.4731, Val Acc: 50.00%
Epoch 13: Tra